# pyscf-cli クイックスタート / Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahntaeyoung1212/pyscf-cli/blob/main/examples/colab_quickstart.ipynb)

XYZファイル1枚とコマンド1行で量子化学計算を実行する教育用CLI **pyscf-cli** を、
インストール不要の Google Colab で体験するノートブックです。
各セルを上から順に実行してください(Shift+Enter)。

*Run real quantum chemistry from a single XYZ file — no local installation needed.*


In [ ]:
# インストール (Colab: 約1-2分) / Install
!pip install -q pyscf-cli || pip install -q git+https://github.com/ahntaeyoung1212/pyscf-cli.git
!pyscf-cli --version


## 1. サンプル分子を取得 / Get a sample molecule


In [ ]:
!pyscf-cli examples          # 一覧を表示
!pyscf-cli examples all      # 全部コピー
!cat h2o.xyz                 # XYZファイルの中身はただのテキスト


## 2. 最初の計算: 水分子のエネルギー / First calculation

RHF/STO-3G の一点計算。出力の **Total Energy**、**MO energies**、**⟨S²⟩** に注目。


In [ ]:
!pyscf-cli energy h2o.xyz


## 3. O₂ のスピン状態 / Spin states of O₂

O₂ の基底状態は三重項(2S+1=3, `--spin 2`)。一重項と比べてみましょう。
⟨S²⟩ = 2.0 が「きれいな三重項」の証拠です。


In [ ]:
!pyscf-cli energy o2.xyz --basis 6-31g                        # 一重項を仮定(誤り!)
!pyscf-cli energy o2.xyz --basis 6-31g --spin 2 --method uhf  # 正しい三重項


## 4. 結合解離曲線 / Bond dissociation (PES scan)


In [ ]:
!pyscf-cli energy h2.xyz --pes --rmin 0.4 --rmax 3.0 --npts 27 --basis 6-31g --json pes.json


In [ ]:
import json
import matplotlib.pyplot as plt
data = json.load(open('pes.json'))
R = [p['R'] for p in data['pes_scan']]
E = [p['e_hartree'] for p in data['pes_scan']]
plt.plot(R, E, 'o-'); plt.xlabel('R (Angstrom)'); plt.ylabel('E (Hartree)')
plt.title('H2 dissociation, RHF/6-31G'); plt.show()


## 5. 構造最適化と振動解析 / Optimize, then vibrations


In [ ]:
!pyscf-cli relax h2o.xyz --basis 6-31g
!pyscf-cli vib h2o-finish.xyz --basis 6-31g --nmax 1


## 6. 振動モードのアニメーション / Vibration movie


In [ ]:
!pyscf-cli vibmovie h2o-finish.xyz --basis 6-31g --nframes 16


In [ ]:
from IPython.display import Image
Image(open('h2o-finish_vibmovie/mode003.gif','rb').read())  # 逆対称伸縮


## 7. 分子の DOS / Molecular DOS plot


In [ ]:
!pyscf-cli dos benzene.xyz --basis 6-31g --align homo --output dos.png --quiet


In [ ]:
from IPython.display import Image
Image('dos.png')


## 8. 熱化学 / Thermochemistry


In [ ]:
!pyscf-cli thermo h2o.xyz --basis 6-31g


## 9. 卒業への一歩: --dry-run / Graduating to PySCF

CLI が裏で何をしているか — 等価な PySCF Python スクリプトを表示します。
これを写して編集すれば、あなたはもう PySCF ユーザーです。


In [ ]:
!pyscf-cli energy h2o.xyz --theory mp2 --basis cc-pvdz --dry-run


## 次に試すこと / What next?

- `!pyscf-cli --help` で全コマンド、`!pyscf-cli info` で基底・汎関数の解説
- `--theory dft --xc b3lyp`、`--theory ccsd_t` など理論レベルを変えてみる
- `orbitals` で HOMO/LUMO の cube を作り、VESTA で可視化
- 演習アイデア集: [docs/TEACHING_ja.md](https://github.com/ahntaeyoung1212/pyscf-cli/blob/main/docs/TEACHING_ja.md)

pyscf-cli は PySCF の非公式教育フロントエンドです。学術利用の際は PySCF を引用してください:
Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020).
